# Force/torque data pre-processing

## 1. Import libraries and helper functions

In [24]:
import sys
sys.path.append('..')
from src import helpers
import numpy as np
import pandas as pd
import os
from tqdm.notebook import  tqdm, tqdm_notebook


In [25]:
from importlib import reload 
reload(helpers)

c:\Users\kmh\Documents\ZF_flight_ontogeny\notebooks\..\src\helpers.py:149: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('takeoff(\d+)', filename) # search for number after 'takeoff' in filename string


<module 'src.helpers' from 'c:\\Users\\kmh\\Documents\\ZF_flight_ontogeny\\notebooks\\..\\src\\helpers.py'>

## 2. Load the force/torque perching data

### Compile the perching force/torque data (subset or whole dataset) into a dataframe
CSV files were generated in Nexus 2.14 using the pipeline "export_csv" (Import ASCII)

In [4]:
csv_df = helpers.compile_FT_CSVs("D:\\Vicon Install\\FinchPerching\\", calibration=False)
#1055 csv files found

Found 1055 .csv files.
Compiled 1055/1055 CSV filepaths in output dataframe.


In [5]:
csv_df.head(5)

,Path,Bird,Date,Filename
0,D:\Vicon Install\FinchPerching\cyanR-blackL_3B...,cyanR-blackL_3B0018D736,20220630_day20_,20220630_day20_104_takeoff2.csv
1,D:\Vicon Install\FinchPerching\cyanR-blackL_3B...,cyanR-blackL_3B0018D736,20220630_day20_,20220630_day20_109_takeoff3.csv
2,D:\Vicon Install\FinchPerching\cyanR-blackL_3B...,cyanR-blackL_3B0018D736,20220630_day20_,20220630_day20_65_takeoff1.csv
3,D:\Vicon Install\FinchPerching\cyanR-blackL_3B...,cyanR-blackL_3B0018D736,20220704_day24_,20220704_day24_12_takeoff1.csv
4,D:\Vicon Install\FinchPerching\cyanR-blackL_3B...,cyanR-blackL_3B0018D736,20220704_day24_,20220704_day24_14_takeoff2.csv


In [6]:
voltage_df = helpers.CSVs_to_dataframe(csv_df, 2022, calibration=False, save_as_csv=False)

  0%|          | 0/1055 [00:00<?, ?it/s]

In [7]:
FT_df = helpers.voltage_to_FT(voltage_df, 2022, calibration='small', cal_data=False)
#something went wrong here. 

In [8]:
print(FT_df.shape) #previously 798720 -> now 83720 -> 496048 (definitely better)
#maybe it was landing number -> one landing -> takeoff number
print(FT_df.dtypes)


(4986048, 12)
FlightID      int64
Bird         object
Age           int32
Takeoff       int64
Note         object
Frame       float64
Fx          float64
Fy          float64
Fz          float64
Tx          float64
Ty          float64
Tz          float64
dtype: object


In [9]:
FT_df

,FlightID,Bird,Age,Takeoff,Note,Frame,Fx,Fy,Fz,Tx,Ty,Tz
0,2302002,cyanR-blackL,20,2,None,1.000,-0.171313,-4.949068,0.222167,-31.343007,-23.632475,4.866516
1,2302002,cyanR-blackL,20,2,None,1.125,-0.178455,-4.961034,0.240858,-31.314601,-23.528739,5.001885
2,2302002,cyanR-blackL,20,2,None,1.250,-0.153476,-4.945042,0.294765,-31.173428,-23.430613,4.356542
3,2302002,cyanR-blackL,20,2,None,1.375,-0.157710,-4.972850,0.298233,-31.358330,-23.241493,4.253488
4,2302002,cyanR-blackL,20,2,None,1.500,-0.175611,-4.970704,0.285146,-31.521466,-23.515666,4.434651
...,...,...,...,...,...,...,...,...,...,...,...,...
4986043,3110012,yellowR-purpleL,100,12,None,480.375,0.304128,-4.623779,0.224969,-34.817840,-13.735589,2.044125
4986044,3110012,yellowR-purpleL,100,12,None,480.500,0.273745,-4.629077,0.225221,-35.107885,-13.777250,2.268091
4986045,3110012,yellowR-purpleL,100,12,None,480.625,0.290768,-4.619870,0.187946,-35.076055,-13.773265,2.756046
4986046,3110012,yellowR-purpleL,100,12,None,480.750,0.295537,-4.643348,0.169920,-35.275260,-13.769305,2.935189


#### (Optional) save dataframe as csv file

In [10]:
# save data (write single csv file) to location below
FT_df.to_csv("C:\\Users\\kmh\\Documents\\DATA\\FT_dataframe_subset_golden8.csv", index=True)
print("CSV file saved!")

CSV file saved!


## 3. Calibrate the data

### 3.1  Calibrate the x-axis
Create new column of Time in seconds (using Frame column as an index) \
t=0 is the time of impact so Time represents time to landing (in seconds) 

In [11]:
FT_df = helpers.calibrate_time(FT_df)
FT_df

,FlightID,Bird,Age,Takeoff,Note,Frame,Fx,Fy,Fz,Tx,Ty,Tz,Time
0,2302002,cyanR-blackL,20,2,None,1.000,-0.171313,-4.949068,0.222167,-31.343007,-23.632475,4.866516,-2.000000
1,2302002,cyanR-blackL,20,2,None,1.125,-0.178455,-4.961034,0.240858,-31.314601,-23.528739,5.001885,-1.998958
2,2302002,cyanR-blackL,20,2,None,1.250,-0.153476,-4.945042,0.294765,-31.173428,-23.430613,4.356542,-1.997917
3,2302002,cyanR-blackL,20,2,None,1.375,-0.157710,-4.972850,0.298233,-31.358330,-23.241493,4.253488,-1.996875
4,2302002,cyanR-blackL,20,2,None,1.500,-0.175611,-4.970704,0.285146,-31.521466,-23.515666,4.434651,-1.995833
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4986043,3110012,yellowR-purpleL,100,12,None,480.375,0.304128,-4.623779,0.224969,-34.817840,-13.735589,2.044125,1.994792
4986044,3110012,yellowR-purpleL,100,12,None,480.500,0.273745,-4.629077,0.225221,-35.107885,-13.777250,2.268091,1.995833
4986045,3110012,yellowR-purpleL,100,12,None,480.625,0.290768,-4.619870,0.187946,-35.076055,-13.773265,2.756046,1.996875
4986046,3110012,yellowR-purpleL,100,12,None,480.750,0.295537,-4.643348,0.169920,-35.275260,-13.769305,2.935189,1.997917


In [14]:
subset_df = FT_df[(FT_df['Bird']== 'cyanR-blackL') & (FT_df['Age'] == 20) & (FT_df['Takeoff'] == 2)]

In [19]:
print(subset_df.head(5))
print(subset_df.shape)

   FlightID          Bird  Age  Takeoff  Note  Frame        Fx        Fy  \
0   2302002  cyanR-blackL   20        2  None  1.000 -0.171313 -4.949068   
1   2302002  cyanR-blackL   20        2  None  1.125 -0.178455 -4.961034   
2   2302002  cyanR-blackL   20        2  None  1.250 -0.153476 -4.945042   
3   2302002  cyanR-blackL   20        2  None  1.375 -0.157710 -4.972850   
4   2302002  cyanR-blackL   20        2  None  1.500 -0.175611 -4.970704   

         Fz         Tx         Ty        Tz      Time  
0  0.222167 -31.343007 -23.632475  4.866516 -2.000000  
1  0.240858 -31.314601 -23.528739  5.001885 -1.998958  
2  0.294765 -31.173428 -23.430613  4.356542 -1.997917  
3  0.298233 -31.358330 -23.241493  4.253488 -1.996875  
4  0.285146 -31.521466 -23.515666  4.434651 -1.995833  
(2880, 13)


In [22]:
#print last recorded timepoint
print(subset_df['Time'].iloc[-1])


0.9989583333333333


### 3.2 Calibrate the force and torques axes
Use the first n frames of each flight sequence as the zero value for a given axis

In [27]:
FT_0centered_df = helpers.zero_centering_takeoff(FT_df, endTime=0.1)
#zero centered by the last 0.1 seconds after takeoff. 

  0%|          | 0/1043 [00:00<?, ?it/s]

In [28]:
print(FT_0centered_df.head(3))
print(FT_0centered_df.dtypes)


   FlightID          Bird  Age  Takeoff  Note  Frame        Fx        Fy  \
0   1202001  greyR-blackL   20        1  None  1.000 -0.136233  0.013110   
1   1202001  greyR-blackL   20        1  None  1.125 -0.173155 -0.011407   
2   1202001  greyR-blackL   20        1  None  1.250 -0.151926 -0.002319   

         Fz        Tx        Ty        Tz      Time  
0 -0.019802 -0.408326 -8.571980  0.398708 -2.000000  
1 -0.005773 -0.034080 -8.589821  0.400655 -1.998958  
2 -0.012530  0.007764 -8.700545  0.114157 -1.997917  
FlightID      int64
Bird         object
Age           int32
Takeoff       int64
Note         object
Frame       float64
Fx          float64
Fy          float64
Fz          float64
Tx          float64
Ty          float64
Tz          float64
Time        float64
dtype: object


In [29]:
FT_0centered_df.shape

(4986048, 13)

## 4. Save the calibrated data to a csv file

In [30]:
FT_0centered_df.to_csv("C:\\Users\\kmh\\Documents\\DATA\\FT_dataframe_subset_golden8_0centered.csv", index=True)
print("centered CSV file saved!")

centered CSV file saved!
